In [ ]:
import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs_Lam0LamC.parquet")
print(list(field for field in data.fields if field[:7]=="LambdaC"))

mothIdx = data['mothIdx']
mcLund = data['mcLund']
d2Idx = data['LambdaCd2Idx']
d2Lund = data['LambdaCd2Lund']
ksMCIdx = data['K_SMCIdx']
kMCIdx = data['KMCIdx']
piMCIdx = data['piMCIdx']
ksFlSig = data['K_SpostFitFlightSignificance']
ksFlBS = data['K_SFlightBSLen']
Bd1Idx = data['Bd1Idx']
lamCMCIdx = data['LambdaCMCIdx']


lamCUncMass = data['LambdaC_unc_Mass']
lamCUncEnergyCM = data['LambdaC_unc_energyCM']
lamCUncp3CM = data['LambdaC_unc_p3CM']



#'nK', 'Kcosth', 'KcosthCM', 'Kenergy', 'KenergyCM', 'Kp3', 'Kp3CM', 'Kphi', 'KphiCM', 'KLund', 'KMCIdx', 'KTrkIdx', 
#'nK_S', 'K_SChi2', 'K_SFlightBSCosA', 'K_SFlightBSErr', 'K_SFlightBSLen', 'K_SFlightCosA', 'K_SFlightErr', 'K_SFlightLen', 'K_SMass', 'K_SMassErr', 'K_SVtxx', 'K_SVtxy', 'K_SVtxz', 'K_Scosth', 'K_ScosthCM', 'K_Senergy', 'K_SenergyCM', 'K_Sp3', 'K_Sp3CM', 'K_Sphi', 'K_SphiCM', 'K_SpostFitFlightSignificance', 'K_SpostFitProbChiSq', 'K_SpreFitMass', 'K_SLund', 'K_SMCIdx', 'K_SVtxStatus', 'K_SnDof', 'K_Sd1Lund', 'K_Sd1Idx', 'K_Sd2Lund', 'K_Sd2Idx'
#K_SFlightBSLen
#K_SpostFitFlightSignificance



In [ ]:
cM = []
tCM = []
flS = []
tFLS = []
for i in range(len(data)):
    for val,d1Val in zip(d2Idx[i][Bd1Idx[i]],Bd1Idx[i]):
        if len(d2Lund[i])>val:
            #K_S decay mode (hopefully)
            if abs(d2Lund[i][val]) == 310: 
                if len(ksFlSig[i])>val:
                    flS += [ksFlSig[i][val]]
                    cM += [lamCUncMass[i][d1Val]]
                    if abs(mcLund[i][mothIdx[i][ksMCIdx[i][val]]])==4122 and abs(mcLund[i][ksMCIdx[i][val]])==310:
                        tCM += [lamCUncMass[i][d1Val]]
                        tFLS += [ksFlSig[i][val]]



In [ ]:
take2CM=[]
take2flS=[]
take2TCM = []
take2TFLS = []
for i in range(len(data)):
    mcLundi = mcLund[i]
    mothIdxi = mothIdx[i]
    first8 = ak.to_list(mcLundi[:8])
    tmB=False
    if 3122 in first8 and 4122 in first8:
        if abs(mcLundi[mothIdxi[first8.index(3122)]])==521 and abs(mcLundi[mothIdxi[first8.index(4122)]])==521:
            tmB = True
    elif -3122 in first8 and -4122 in first8:
        if abs(mcLundi[mothIdxi[first8.index(-3122)]])==521 and abs(mcLundi[mothIdxi[first8.index(-4122)]])==521:
            tmB=True

    if tmB:
        #truth matched B
        Bd1Idxi = Bd1Idx[i]
        d2Idxi = d2Idx[i]
        ksFlSigi = ksFlSig[i]
        ksMCIdxi = ksMCIdx[i]
        d2Lundi = d2Lund[i]
        lamCUncMassi = lamCUncMass[i]
        for d1Val in Bd1Idxi:
            for val in d2Idxi[Bd1Idxi]:
                #print(val,d1Val)
                if len(d2Lundi)>val:
                    #K_S decay mode (hopefully)
                    if abs(d2Lundi[val]) == 310: 
                        if len(ksFlSigi)>val:
                            if abs(mcLundi[mothIdxi[ksMCIdxi[val]]])==4122 and abs(mcLundi[ksMCIdxi[val]])==310:
                                take2TCM += [lamCUncMassi[d1Val]]
                                take2TFLS += [ksFlSigi[val]]
                            else:
                                take2flS += [ksFlSigi[val]]
                                take2CM += [lamCUncMassi[d1Val]]

In [ ]:
fls=take2flS
tFLS = take2TFLS
cM = take2CM
tCM = take2TCM

print(len(fls))
print(len(tFLS))
print(len(cM))
print(len(tCM))

plt.figure()
counts,bins,_=plt.hist(fls, bins=80, range=(0,400),density=False,label = "Non truth matched K_S in same event", histtype="step",linewidth=2)
print(counts.sum())
counts,bins,_=plt.hist(tFLS, bins=40, range=(0,400),density=False,label = "Truth Matched K_S", histtype="step",linewidth=2)
print(counts.sum())
plt.xlabel("K_S FlightSig")
plt.legend()
plt.show()

plt.figure()
counts,bins,_=plt.hist(cM,bins=40, density=False,label = "Non truth matched Lam C in same event", histtype="step",linewidth=2)
print(counts.sum())
counts,bins,_=plt.hist(tCM,bins=40, density=False,label = "Truth Matched LamC", histtype="step",linewidth=2)
print(counts.sum())
plt.xlabel("LamC Mass")
plt.legend()
plt.show()



In [ ]:
plt.figure(figsize = (8,6))
counts, xedges, yedges, image = plt.hist2d(
        fls, 
        cM,
        bins=200, 
        range=[[-0.2,500],[2.22,2.37]], 
        cmap='viridis'
    )

plt.colorbar(label='Candidates per bin')
plt.ylabel("LamCUncMass")
plt.xlabel("K_S Dau Flight Significance")
plt.title('2D plot! Non truth match')

plt.legend() 
plt.show()

x_lo = np.searchsorted(xedges, 0, side="left")
x_hi = np.searchsorted(xedges, 5, side="right") - 1
y_lo = np.searchsorted(yedges, 2.22, side="left")
y_hi = np.searchsorted(yedges, 2.37, side="right") - 1

region_sum = counts[x_lo:x_hi, y_lo:y_hi].sum()

print("Count in FlSig = [0,5]: ", region_sum)
print("Total Count:", counts.sum())
print("Percentage:", region_sum/counts.sum()*100,"%")

#2.27979 : 2.29149
x_lo = np.searchsorted(xedges, 0, side="left")
x_hi = np.searchsorted(xedges, 500, side="right") - 1
y_lo = np.searchsorted(yedges, 2.27979, side="left")
y_hi = np.searchsorted(yedges, 2.29149, side="right") - 1

region_sum = counts[x_lo:x_hi, y_lo:y_hi].sum()

print("Count in UncMass = [2.27979, 2.29149]: ", region_sum)
print("Total Count:", counts.sum())
print("Percentage:", region_sum/counts.sum()*100,"%")

x_lo = np.searchsorted(xedges, 5, side="left")
x_hi = np.searchsorted(xedges, 500, side="right") - 1
y_lo = np.searchsorted(yedges, 2.27979, side="left")
y_hi = np.searchsorted(yedges, 2.29149, side="right") - 1

region_sum = counts[x_lo:x_hi, y_lo:y_hi].sum()

print("Count in FlSig [5,500], UncMass [2.27979, 2.29149]: ", region_sum)
print("Total Count:", counts.sum())
print("Percentage:", region_sum/counts.sum()*100,"%")

plt.figure(figsize = (8,6))
counts, xedges, yedges, image = plt.hist2d(
        tFLS, 
        tCM,
        bins=200, 
        range=[[-0.2,500],[2.22,2.37]], 
        cmap='viridis'
    )

plt.colorbar(label='Candidates per bin')
plt.ylabel("LamCUncMass")
plt.xlabel("K_S Dau Flight Significance")
plt.title('2D plot! Truth Matched K_S')

plt.legend() 
plt.show()

x_lo = np.searchsorted(xedges, 0, side="left")
x_hi = np.searchsorted(xedges, 5, side="right") - 1
y_lo = np.searchsorted(yedges, 2.22, side="left")
y_hi = np.searchsorted(yedges, 2.37, side="right") - 1

region_sum = counts[x_lo:x_hi, y_lo:y_hi].sum()

print("Count in FlSig = [0,5]: ", region_sum)
print("Total Count:", counts.sum())
print("Percentage:", region_sum/counts.sum()*100,"%")

#2.27979 : 2.29149
x_lo = np.searchsorted(xedges, 0, side="left")
x_hi = np.searchsorted(xedges, 500, side="right") - 1
y_lo = np.searchsorted(yedges, 2.27979, side="left")
y_hi = np.searchsorted(yedges, 2.29149, side="right") - 1

region_sum = counts[x_lo:x_hi, y_lo:y_hi].sum()

print("Count in UncMass = [2.27979, 2.29149]: ", region_sum)
print("Total Count:", counts.sum())
print("Percentage:", region_sum/counts.sum()*100,"%")

x_lo = np.searchsorted(xedges, 5, side="left")
x_hi = np.searchsorted(xedges, 500, side="right") - 1
y_lo = np.searchsorted(yedges, 2.27979, side="left")
y_hi = np.searchsorted(yedges, 2.29149, side="right") - 1

region_sum = counts[x_lo:x_hi, y_lo:y_hi].sum()

print("Count in FlSig [5,500], UncMass [2.27979, 2.29149]: ", region_sum)
print("Total Count:", counts.sum())
print("Percentage:", region_sum/counts.sum()*100,"%")


In [ ]:

fx = []
fxSE = []
fFake= []
fFakeSE = []

bx = []
bxSE = []
bFake = []
bFakeSE = []
trueLongCount = 0
fakeShortCount = 0
trueShortCount = 0
fakeLongCount = 0
trueLongAndShortCount = 0
neitherCount = 0
k_sDuaAndMultKSCount = 0

dLD = {0:lamCUncMass, 1:lamCUncEnergyCM, 2:lamCUncp3CM}
labelD = {0: "lamCUncMass", 1:"lamCUncEnergyCM", 2:"lamCUncp3CM"}
rangeD = {0: (2.22,2.35),1:(2.6,3.6),2:(1.3,2.75)}
countsXD = {}
countsYD = {}
binsXD = {}
binsYD = {}

#(2.22,2.35)

for dL in [0]:
    x = []
    y = []
    trueLamCount = 0
    fakeLamCount = 0
    checkedE = 0
    hmCount = 0
    for i in range(len(data)):
        lcM_i = dLD[dL][i]
        if len(lcM_i)>1 and (len(set(ak.to_list(lamCMCIdx[i])))>1 or -1 not in lamCMCIdx[i]):
            checkedE += 1
            #closestInd = min(range(len(lcM_i)), key=lambda idx: abs(lcM_i[idx] - 2.286))
            #x += [lcM_i[closestInd]]
            #y += [lcM_i[idx] for idx in range(len(lcM_i)) if idx != closestInd]
            trueLamL = [idx for idx in set(ak.to_list(Bd1Idx[i])) if abs(mcLund[i][lamCMCIdx[i][idx]]) == 4122]
            trueLamCount += len(trueLamL)
            fakeLamCount += len(lamCMCIdx[i]) - len(trueLamL)
            x += [lcM_i[val] for val in trueLamL]
            y += [lcM_i[val] for val in set(ak.to_list(Bd1Idx[i])) if val not in trueLamL]


    plt.figure()
    counts, bins, _ = plt.hist(x, bins = 100, range=rangeD[dL],density=False,label = "True LamC", histtype="step",linewidth=2)
    countsY, binsY, _ = plt.hist(y, bins = 100, range=rangeD[dL],density=False, color="orange", label = "All other LambdaC", histtype="step",linewidth=2)
    plt.xlabel(labelD[dL])
    plt.legend()
    plt.show()

    countsXD[dL] = counts
    countsYD[dL] = countsY

    binsXD[dL] = bins
    binsYD[dL] = binsY



    print(len(x), len(y))
    print(np.sum(counts), np.sum(countsY))





"""
for i in range(len(data)):
    if len(ksMCIdx[i])>1:
        longFS = 0
        shortFS = 1000
        ksDauB = False
        ksDauL = []

        for indD in range(len(d2Idx[i])):
            if len(ksMCIdx[i])>indD:
                if abs(mcLund[i][mothIdx[i][ksMCIdx[i][indD]]]) == 4122:
                    ksDauB = True
                    ksDauL += [indD]

        if ksDauB:
            k_sDuaAndMultKSCount+=1

            longFS = max(range(len(ksFlSig[i])), key=ksFlSig[i].__getitem__)
            shortFS = min(range(len(ksFlSig[i])), key=ksFlSig[i].__getitem__)

            longBool = longFS in ksDauL
            shortBool = shortFS in ksDauL


            if longBool and not shortBool:
                trueLongCount += 1
                fakeShortCount+=1
            elif shortBool and not longBool:
                trueShortCount+=1
                fakeLongCount+=1
            elif longBool and shortBool:
                trueLongAndShortCount += 1
                trueLongCount += 1
                trueShortCount += 1
            else:
                neitherCount += 1
                fakeLongCount+=1
                fakeShortCount+=1

    if i%10000 == 0:
        print(i)


print(f"Total events considered (multiple KSMCIdx and a K_s daughter of LamC): {k_sDuaAndMultKSCount}")
print(f"Number, % of true K_S with longest FlightSig: {trueLongCount}, {trueLongCount/k_sDuaAndMultKSCount*100}%")
print(f"Number, % of true K_S with shortest FlightSig: {trueShortCount}, {trueShortCount/k_sDuaAndMultKSCount*100}%")
print()
print(f"Number, % of fake K_S with longest FlightSig: {fakeLongCount}, {fakeLongCount/k_sDuaAndMultKSCount*100}%")
print(f"Number, % of fake K_S with shortest FlightSig: {fakeShortCount}, {fakeShortCount/k_sDuaAndMultKSCount*100}%")
print()
print(f"Number,% that both longest and shortest flightSig were True: {trueLongAndShortCount}, {trueLongAndShortCount/k_sDuaAndMultKSCount*100}%")
print(f"Number,% that neither longest nor shortest flightSig were True: {neitherCount}, {neitherCount/k_sDuaAndMultKSCount*100}%")
"""
    
"""for i in range(len(data)):
    #print("I:", i)
    for val in set(ak.to_list(d2Idx[i])):
        #print("d2Idx val:", val)
        if len(ksMCIdx[i]) > val:
            #print("K_S")
            if -1 not in ksMCIdx[i]:
                if len(ksMCIdx[i])>1:
                    if abs(mcLund[i][mothIdx[i][ksMCIdx[i][val]]]) == 4122 or (abs(mcLund[i][mothIdx[i][ksMCIdx[i][val]]]) == 311 and abs(mcLund[i][mothIdx[i][mothIdx[i][ksMCIdx[i][val]]]])) == 4122:
                        fxSE += [ksFlSig[i][val]]
                        bxSE += [ksFlBS[i][val]]
                        fx += [ksFlSig[i][val]]
                        bx += [ksFlBS[i][val]]
                    else:
                        fFakeSE += [ksFlSig[i][val]]
                        bFakeSE += [ksFlBS[i][val]]
                        fFake += [ksFlSig[i][val]]
                        bFake += [ksFlBS[i][val]]

                else:
                    if abs(mcLund[i][mothIdx[i][ksMCIdx[i][val]]]) == 4122 or (abs(mcLund[i][mothIdx[i][ksMCIdx[i][val]]]) == 311 and abs(mcLund[i][mothIdx[i][mothIdx[i][ksMCIdx[i][val]]]])) == 4122:
                        fx += [ksFlSig[i][val]]
                        bx += [ksFlBS[i][val]]
                    else:
                        fFake += [ksFlSig[i][val]]
                        bFake += [ksFlBS[i][val]]


            else:
                #print(ksMCIdx[i])
                #print("Fake KS")
                #print("FLSIG", ksFlSig[i])
                #print("BS", ksFlBS[i])

                fFake += [ksFlSig[i][val]]
                bFake += [ksFlBS[i][val]]

            
"""
#for dBool in [False]:
if False:
    plt.figure()
    plt.hist(fx, bins = 100, density=dBool, range = (0,400), label = "Truth K_S", histtype="step",linewidth=2)
    plt.hist(fFake, bins = 100, density=dBool, color="orange",range = (0,400), label = "Fake K_S", histtype="step",linewidth=2)
    plt.xlabel(f"K_SpostFitFlightSignificance, density = {dBool}")
    plt.legend()
    plt.show()

    plt.figure()
    plt.hist(fx, bins = 40, density=dBool, range = (0,150), label = "Truth K_S", histtype="step",linewidth=2)
    plt.hist(fFake, bins = 40, density=dBool, color="orange",range = (0,150), label = "Fake K_S", histtype="step",linewidth=2)
    plt.xlabel(f"K_SpostFitFlightSignificance, density = {dBool}")
    plt.legend()
    plt.show()


    plt.figure()
    plt.hist(fxSE, bins = 100, density=dBool, range = (0,400), label = "Truth K_S", histtype="step",linewidth=2)
    plt.hist(fFakeSE, bins = 100, density=dBool, color="orange",range = (0,400), label = "Fake K_S, same event", histtype="step",linewidth=2)
    plt.xlabel(f"K_SpostFitFlightSignificance, same Event, , density = {dBool}")
    plt.legend()
    plt.show()

    """plt.figure()
    plt.hist(bx, bins = 30, range = (0,105),density=False, label = "Truth K_S", histtype="step",linewidth=2)
    plt.hist(bFake, bins = 30,range = (0,105), density=False, color="orange", label = "Fake K_S", histtype="step",linewidth=2)
    plt.xlabel("K_SFlightBSLen")
    plt.legend()
    plt.show()"""

    plt.figure()
    plt.hist(bx, bins = 40, range = (0,40),density=dBool, label = "Truth K_S", histtype="step",linewidth=2)
    plt.hist(bFake, bins = 40,range = (0,40), density=dBool, color="orange", label = "Fake K_S", histtype="step",linewidth=2)
    plt.xlabel(f"K_SFlightBSLen, density = {dBool}")
    plt.legend()
    plt.show()

    plt.figure()
    plt.hist(bxSE, bins = 40, density=dBool, range = (0,40), label = "Truth K_S", histtype="step",linewidth=2)
    plt.hist(bFakeSE, bins = 40, density=dBool, color="orange",range = (0,40), label = "Fake K_S, same event", histtype="step",linewidth=2)
    plt.xlabel(f"K_SFlightBSLen, same Event, density = {dBool}")
    plt.legend()
    plt.show()

In [ ]:
import statistics
med = statistics.median(x)

rangeC = 20
for midPointDist in range(rangeC):
    middle = med-(rangeC/2/1000)+midPointDist/1000

    for halfDist in range(30):
        cutVal = halfDist/1000
        cutX = list(val for val in x if middle-cutVal<=val<=middle+cutVal)
        cutY = list(val for val in y if middle-cutVal<=val<=middle+cutVal)
        if 0.95*len(x)<=len(cutX)<=0.9525*len(x):
            print("Fake % in range:", len(cutY)/len(y)*100, "% / LowerBound:", middle-cutVal, "/ UpperBound:", middle+cutVal, "/ Middle:", middle, "/ True Med:", med, "/ Count:", len(cutX), "/ percentage:", len(cutX)/len(x)*100,"% / 95% bound:", 0.95*len(x), "/ Total Count:", len(x))

In [ ]:
for dL in range(3):
    for choice in range(2):
        bins = binsXD[dL]
        counts = countsXD[dL]
        countsY = countsYD[dL]
        
        binW = bins[1]-bins[0]

        plt.figure()
        for binCount in range(5,80):
            xpc = []
            xC=[]
            yC=[]
            ypc = []
            xval = []
            for binMin in range(10, 90):
                binMax = binMin+binCount
                xval += [rangeD[dL][0]+binMin*binW]
                xpc+=[np.sum(counts[binMin:binMax])/np.sum(counts)*100]
                ypc+=[np.sum(countsY[binMin:binMax])/np.sum(countsY)*100]
                xC+=[np.sum(counts[binMin:binMax])]
                yC+=[np.sum(countsY[binMin:binMax])]

            pcDiv = list(x/y for x,y in zip(xpc,ypc))
            cDiv = list(x/np.sqrt(y) for x,y in zip(xC,yC))
            print(f"{binCount}, MAX pcDiv: {max(pcDiv)}, MAX cDiv: {max(cDiv)}")
            print(f"PCDiv bounds: [{xval[pcDiv.index(max(pcDiv))]:.5f} : {xval[pcDiv.index(max(pcDiv))]+binCount*binW:.5f}]")
            print(f"CDiv bounds: [{xval[cDiv.index(max(cDiv))]:.5f} : {xval[cDiv.index(max(cDiv))]+binCount*binW:.5f}]")

            if choice == 0:
                plt.plot(xval, pcDiv, label=f"bins={binCount}")
            else:
                plt.plot(xval, cDiv, label=f"bins={binCount}")
            
        if choice == 0:
            plt.title(f"x%/y% versus minimum bins (binWidth={binW:.5f})")
        else:
            plt.title(f"x/sqrt(y) versus min mass (binWidth={binW:.5f}, values = {labelD[dL]}")
        #plt.legend()
        plt.show()


In [ ]:
optD = {0: [2.27979,2.29149], 1: [2.90000,3.15000], 2:[1.79300,2.17000]}
for dL in range(2):
    for dL2 in range(dL+1,3):
        x = []
        y = []
        for i in range(len(data)):
            if -1 in set(ak.to_list(lamCMCIdx[i])):
                lcM_i = dLD[dL][i]
                lcM_i2 = dLD[dL2][i]
                for val in set(list(value for value in Bd1Idx[i] if lamCMCIdx[i][value]==-1)):
                    x += [lcM_i[val]]
                    y += [lcM_i2[val]]
            





        plt.figure(figsize = (8,6))
        counts, xedges, yedges, image = plt.hist2d(
                x, 
                y,
                bins=100, 
                range=[rangeD[dL], rangeD[dL2]], 
                cmap='viridis'
            )

        
        x_lo = np.searchsorted(xedges, optD[dL][0], side="left")
        x_hi = np.searchsorted(xedges, optD[dL][1], side="right") - 1
        y_lo = np.searchsorted(yedges, optD[dL2][0], side="left")
        y_hi = np.searchsorted(yedges, optD[dL2][1], side="right") - 1

        region_sum = counts[x_lo:x_hi, y_lo:y_hi].sum()

        print("SIGNAL COUNT:", region_sum)
        print("Total Count:", counts.sum())
        print("Percentage:", region_sum/counts.sum()*100,"%")

        plt.colorbar(label='Candidates per bin')
        plt.xlabel(labelD[dL])
        plt.ylabel(labelD[dL2])
        plt.title('2D plot!')

        plt.legend() 
        plt.show()